# Frozen-feature generalization probes

Inspect LOPO clustered intervals and five-seed repartition results for lightweight frozen-feature probes. These are not the fine-tuned main encoder experiments.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## LOPO held-out platform estimates

In [ ]:
l=pd.read_csv(ROOT / "results/generalization/LOPO_CLUSTER_BOOTSTRAP_CI.csv")
display(l[l.model.eq("siglip2_vta_probe")][["held_out_platform","n_windows","n_groups","macro_f1","cluster_ci_low","cluster_ci_high"]])

## Five frozen-feature group repartitions

In [ ]:
m=pd.read_csv(ROOT / "results/generalization/REPEATED_SPLIT_METRICS.csv")
summ=pd.read_csv(ROOT / "results/generalization/REPEATED_SPLIT_SUMMARY.csv")
display(summ)
assert set(m.seed)=={42,2026,2027,1301,1302}
assert len(m)==20
check=m.groupby("model").agg(macro_f1_mean=("macro_f1","mean"),roc_auc_mean=("roc_auc","mean")).reset_index().merge(summ,on="model")
assert np.allclose(check.macro_f1_mean_x,check.macro_f1_mean_y)
assert np.allclose(check.roc_auc_mean_x,check.roc_auc_mean_y)
print("PASS: stored five-seed summary matches individual metrics")

## Descriptive Macro F1 by split

In [ ]:
fig,ax=plt.subplots(figsize=(7,3.7))
for name,g in m.groupby("model"):
    ax.plot(g.seed.astype(str),g.macro_f1,marker="o",label=name)
ax.set(xlabel="Group-partition seed",ylabel="Macro F1",title="Frozen-feature probes (overlapping test partitions)")
ax.legend(loc="best",fontsize=8)
plt.tight_layout();plt.show()